In [ ]:
# =================================================================
# PREPARACIÓN DE DATOS PARA MODELADO
# =================================================================
# Objetivo: Generar CSVs limpios listos para análisis y modelado
# Input: dataset_etiquetado_tesis.csv + Excel VAB del BCE
# Output: CSVs preparados para modelado
# =================================================================
# CONFIGURACIÓN INICIAL
# =================================================================

import pandas as pd
import numpy as np
import os
from google.colab import drive
import warnings
warnings.filterwarnings('ignore')

# Montar Google Drive
drive.mount('/content/drive')

# CONFIGURACIÓN
path_proyecto = "/content/drive/MyDrive/TitulacionF"

print("✅ Librerías cargadas y Drive montado")
print(f"📁 Ruta del proyecto: {path_proyecto}")




Mounted at /content/drive
✅ Librerías cargadas y Drive montado
📁 Ruta del proyecto: /content/drive/MyDrive/TitulacionF


In [ ]:
# =================================================================
# CARGAR Y EXPLORAR DATASET DE NOTICIAS
# =================================================================

# Cargar dataset etiquetado
ruta_noticias = f"{path_proyecto}/dataset_etiquetado_tesis.csv"

if not os.path.exists(ruta_noticias):
    print("❌ ERROR: No se encuentra dataset_etiquetado_tesis.csv")
    print("Ejecuta primero el Notebook 1 (scraping y etiquetado)")
else:
    df_noticias = pd.read_csv(ruta_noticias)

    print("✅ Dataset de noticias cargado")
    print(f"\n📊 INFORMACIÓN GENERAL:")
    print(f"  • Total registros: {len(df_noticias)}")
    print(f"  • Columnas: {list(df_noticias.columns)}")

    # Convertir fecha a datetime
    if df_noticias['fecha'].dtype == 'object':
        df_noticias['fecha'] = pd.to_datetime(df_noticias['fecha'])

    # Crear columna año y mes
    if 'año' not in df_noticias.columns:
        df_noticias['año'] = df_noticias['fecha'].dt.year
    if 'mes' not in df_noticias.columns:
        df_noticias['mes'] = df_noticias['fecha'].dt.month

    # Crear columna año_mes para agregación
    df_noticias['año_mes'] = df_noticias['fecha'].dt.to_period('M').astype(str)

    print(f"\n📅 Rango temporal:")
    print(f"  • Desde: {df_noticias['fecha'].min()}")
    print(f"  • Hasta: {df_noticias['fecha'].max()}")
    print(f"  • Años presentes: {sorted(df_noticias['año'].unique())}")

    print(f"\n🎭 Distribución de sentimientos:")
    print(df_noticias['sentimiento'].value_counts())

    # Mostrar ejemplos
    print(f"\n📰 EJEMPLOS DE NOTICIAS:")
    print("-" * 70)
    for idx, row in df_noticias.head(3).iterrows():
        print(f"\n[{row['sentimiento']}] {row['fecha'].date()}")
        print(f"{row['texto'][:150]}...")


✅ Dataset de noticias cargado

📊 INFORMACIÓN GENERAL:
  • Total registros: 12180
  • Columnas: ['fecha', 'texto', 'url', 'fuente', 'consulta', 'texto_limpio', 'num_palabras', 'sentimiento']

📅 Rango temporal:
  • Desde: 2020-01-04 08:00:00
  • Hasta: 2026-03-16 06:54:47
  • Años presentes: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

🎭 Distribución de sentimientos:
sentimiento
Positivo    5287
Neutro      5149
Negativo    1744
Name: count, dtype: int64

📰 EJEMPLOS DE NOTICIAS:
----------------------------------------------------------------------

[Positivo] 2025-12-04
Construyen el mall más grande del mundo: costará 100 millones de dólares y tendrá 180 tiendas - El Cronista. Construyen el mall más grande del mundo: ...

[Neutro] 2026-03-08
Portoviejo ya no figura entre las diez economías urbanas más grandes del país - eldiario.ec. Portoviejo ya no figura entre las diez economías urbanas ...

[Neutro] 2025-08-05
Liga 

In [ ]:
# =================================================================
# FILTRAR DATOS 2020-2025
# =================================================================

print("\n" + "=" * 70)
print("FILTRADO DE DATOS 2020-2024")
print("=" * 70)

# Filtrar solo años 2020-2025
df_filtrado = df_noticias[df_noticias['año'].between(2020, 2026)].copy()

print(f"\n📊 Resultados del filtrado:")
print(f"  • Registros originales: {len(df_noticias)}")
print(f"  • Registros 2020-2024: {len(df_filtrado)}")
print(f"  • Registros eliminados: {len(df_noticias) - len(df_filtrado)}")

# Distribución por año
print(f"\n📅 Distribución por año (2020-2024):")
dist_anual = df_filtrado['año'].value_counts().sort_index()
for año, count in dist_anual.items():
    print(f"  • {año}: {count} noticias")

# Distribución de sentimientos en período filtrado
print(f"\n🎭 Sentimientos en 2020-2024:")
dist_sent = df_filtrado['sentimiento'].value_counts()
total = len(df_filtrado)
for sent, count in dist_sent.items():
    pct = (count/total)*100
    print(f"  • {sent}: {count} ({pct:.1f}%)")


FILTRADO DE DATOS 2020-2024

📊 Resultados del filtrado:
  • Registros originales: 12180
  • Registros 2020-2024: 12180
  • Registros eliminados: 0

📅 Distribución por año (2020-2024):
  • 2020: 509 noticias
  • 2021: 820 noticias
  • 2022: 1065 noticias
  • 2023: 1336 noticias
  • 2024: 2193 noticias
  • 2025: 4831 noticias
  • 2026: 1426 noticias

🎭 Sentimientos en 2020-2024:
  • Positivo: 5287 (43.4%)
  • Neutro: 5149 (42.3%)
  • Negativo: 1744 (14.3%)


In [ ]:
# =================================================================
# CONVERTIR SENTIMIENTO A VALOR NUMÉRICO
# =================================================================

print("\n" + "=" * 70)
print("CONVERSIÓN DE SENTIMIENTO A VALOR NUMÉRICO")
print("=" * 70)

# Mapeo de sentimiento a escala numérica
mapeo_sentimiento = {
    'Positivo': 1,
    'Neutro': 0,
    'Negativo': -1
}

df_filtrado['sentimiento_score'] = df_filtrado['sentimiento'].map(mapeo_sentimiento)

print("\n✅ Sentimientos convertidos a escala numérica:")
print("  • Positivo → +1")
print("  • Neutro → 0")
print("  • Negativo → -1")

# Verificar conversión
print(f"\n📊 Estadísticas del sentimiento numérico:")
print(df_filtrado['sentimiento_score'].describe())

# Ejemplos
print(f"\n📝 Ejemplos de conversión:")
muestra = df_filtrado[['fecha', 'sentimiento', 'sentimiento_score']].head(5)
print(muestra.to_string(index=False))


CONVERSIÓN DE SENTIMIENTO A VALOR NUMÉRICO

✅ Sentimientos convertidos a escala numérica:
  • Positivo → +1
  • Neutro → 0
  • Negativo → -1

📊 Estadísticas del sentimiento numérico:
count    12180.000000
mean         0.290887
std          0.701914
min         -1.000000
25%          0.000000
50%          0.000000
75%          1.000000
max          1.000000
Name: sentimiento_score, dtype: float64

📝 Ejemplos de conversión:
              fecha sentimiento  sentimiento_score
2025-12-04 08:00:00    Positivo                  1
2026-03-08 14:35:00      Neutro                  0
2025-08-05 07:00:00      Neutro                  0
2026-03-03 08:00:00    Positivo                  1
2023-01-18 08:00:00    Negativo                 -1


In [ ]:
# =================================================================
# PREPARAR SENTIMIENTO MENSUAL 2020-2024 (CON FEATURES AVANZADOS)
# =================================================================

print("\n" + "=" * 70)
print("PREPARACIÓN SENTIMIENTO MENSUAL 2020-2024 + FEATURES")
print("=" * 70)

# Usar df_filtrado
print(f"\n📊 Datos a procesar:")
print(f"  • Años: 2020-2024")
print(f"  • Total noticias: {len(df_filtrado)}")

# Agregar por mes (usando sentimiento_score numérico)
sentimiento_mensual_completo = df_filtrado.groupby('año_mes').agg({
    'sentimiento_score': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()

# Aplanar columnas multi-nivel
sentimiento_mensual_completo.columns = ['año_mes', 'sentimiento_promedio', 'sentimiento_std',
                                         'sentimiento_min', 'sentimiento_max', 'num_noticias']

# Agregar año y mes
año_mes_info = df_filtrado.groupby('año_mes')[['año', 'mes']].first().reset_index()
sentimiento_mensual_completo = sentimiento_mensual_completo.merge(año_mes_info, on='año_mes')

# Calcular proporciones de cada sentimiento
conteo_sentimientos = df_filtrado.groupby(['año_mes', 'sentimiento']).size().unstack(fill_value=0)

# Agregar conteos absolutos
if 'Positivo' in conteo_sentimientos.columns:
    sentimiento_mensual_completo['num_positivas'] = sentimiento_mensual_completo['año_mes'].map(conteo_sentimientos['Positivo'])
else:
    sentimiento_mensual_completo['num_positivas'] = 0

if 'Negativo' in conteo_sentimientos.columns:
    sentimiento_mensual_completo['num_negativas'] = sentimiento_mensual_completo['año_mes'].map(conteo_sentimientos['Negativo'])
else:
    sentimiento_mensual_completo['num_negativas'] = 0

if 'Neutro' in conteo_sentimientos.columns:
    sentimiento_mensual_completo['num_neutras'] = sentimiento_mensual_completo['año_mes'].map(conteo_sentimientos['Neutro'])
else:
    sentimiento_mensual_completo['num_neutras'] = 0

# Rellenar NaN con 0
sentimiento_mensual_completo[['num_positivas', 'num_negativas', 'num_neutras']] = \
    sentimiento_mensual_completo[['num_positivas', 'num_negativas', 'num_neutras']].fillna(0)

# Calcular porcentajes
sentimiento_mensual_completo['pct_positivas'] = (sentimiento_mensual_completo['num_positivas'] /
                                                   sentimiento_mensual_completo['num_noticias'] * 100).round(1)
sentimiento_mensual_completo['pct_negativas'] = (sentimiento_mensual_completo['num_negativas'] /
                                                   sentimiento_mensual_completo['num_noticias'] * 100).round(1)
sentimiento_mensual_completo['pct_neutras'] = (sentimiento_mensual_completo['num_neutras'] /
                                                 sentimiento_mensual_completo['num_noticias'] * 100).round(1)

# ===== FEATURES ADICIONALES PARA CALIBRACIÓN =====

# 1. Volatilidad móvil (ventana 3 meses)
sentimiento_mensual_completo['volatilidad_movil_3m'] = (
    sentimiento_mensual_completo['sentimiento_promedio'].rolling(window=3, min_periods=1).std()
)

# 2. Tendencia (cambio respecto al mes anterior)
sentimiento_mensual_completo['cambio_mes_anterior'] = (
    sentimiento_mensual_completo['sentimiento_promedio'].diff()
)

# 3. Media móvil 3 meses
sentimiento_mensual_completo['media_movil_3m'] = (
    sentimiento_mensual_completo['sentimiento_promedio'].rolling(window=3, min_periods=1).mean()
)

# 4. Indicador de momentum (cambio acelerado)
sentimiento_mensual_completo['momentum'] = (
    sentimiento_mensual_completo['cambio_mes_anterior'].diff()
)

# 5. Ratio positivo/negativo (evitando división por cero)
sentimiento_mensual_completo['ratio_pos_neg'] = np.where(
    sentimiento_mensual_completo['num_negativas'] > 0,
    sentimiento_mensual_completo['num_positivas'] / sentimiento_mensual_completo['num_negativas'],
    sentimiento_mensual_completo['num_positivas']
)

print("\n✅ Features adicionales creados:")
print("  • Volatilidad móvil 3 meses")
print("  • Cambio mes anterior (tendencia)")
print("  • Media móvil 3 meses")
print("  • Momentum (aceleración)")
print("  • Ratio positivo/negativo")

# Guardar dataset mensual COMPLETO (2020-2024)
ruta_mensual_completo = f"{path_proyecto}/sentimiento_mensual_2020_2024.csv"
sentimiento_mensual_completo.to_csv(ruta_mensual_completo, index=False, encoding='utf-8-sig')

print(f"\n💾 Dataset mensual guardado:")
print(f"📁 {ruta_mensual_completo}")
print(f"📊 {len(sentimiento_mensual_completo)} registros mensuales (2020-2024)")

# Mostrar distribución por año
print(f"\n📈 DISTRIBUCIÓN POR AÑO:")
print("-" * 70)
resumen_anual = sentimiento_mensual_completo.groupby('año').agg({
    'sentimiento_promedio': 'mean',
    'num_noticias': 'sum',
    'pct_positivas': 'mean',
    'pct_negativas': 'mean',
    'volatilidad_movil_3m': 'mean'
}).round(3)
print(resumen_anual)

# Mostrar primeros y últimos meses
print(f"\n📅 PRIMEROS MESES 2020:")
print(sentimiento_mensual_completo[sentimiento_mensual_completo['año'] == 2020].head(3)[
    ['año_mes', 'sentimiento_promedio', 'num_noticias', 'volatilidad_movil_3m', 'cambio_mes_anterior']
].to_string(index=False))

print(f"\n📅 ÚLTIMOS MESES 2024:")
print(sentimiento_mensual_completo[sentimiento_mensual_completo['año'] == 2024].tail(3)[
    ['año_mes', 'sentimiento_promedio', 'num_noticias', 'volatilidad_movil_3m', 'cambio_mes_anterior']
].to_string(index=False))


PREPARACIÓN SENTIMIENTO MENSUAL 2020-2024 + FEATURES

📊 Datos a procesar:
  • Años: 2020-2024
  • Total noticias: 12180

✅ Features adicionales creados:
  • Volatilidad móvil 3 meses
  • Cambio mes anterior (tendencia)
  • Media móvil 3 meses
  • Momentum (aceleración)
  • Ratio positivo/negativo

💾 Dataset mensual guardado:
📁 /content/drive/MyDrive/TitulacionF/sentimiento_mensual_2020_2024.csv
📊 75 registros mensuales (2020-2024)

📈 DISTRIBUCIÓN POR AÑO:
----------------------------------------------------------------------
      sentimiento_promedio  num_noticias  pct_positivas  pct_negativas  \
año                                                                      
2020                 0.261           509         40.467         14.375   
2021                 0.312           820         44.000         12.800   
2022                 0.358          1065         47.183         11.417   
2023                 0.313          1336         43.492         12.192   
2024                 0.2

In [ ]:
# =================================================================
# CREAR VAB MANUALMENTE
# =================================================================

print("\n" + "=" * 70)
print("INGRESAR DATOS VAB MANUALMENTE")
print("=" * 70)

datos_vab_ejemplo = {
    'año': [2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
            2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'vab_corriente': [
        2077.7, 2603.6, 2729.4, 2969.4, 3797.4, 4178.01, 4369.5,
        4533.8, 4786.1, 5061.6, 4998.03, 3820.5, 4898.2, 4515.5,
        4906.4, 5496.6, 5903.5
    ]
}

print("\n```python")
print("datos_vab_manabi = {")
print("    'año': [2007, 2008, ..., 2023],")
print("    'vab_corriente': [valor1, valor2, ..., valorN]")
print("}")
print("df_vab_manabi = pd.DataFrame(datos_vab_manabi)")
print("```")


df_vab_manabi = pd.DataFrame(datos_vab_ejemplo)
print("\n✅ Datos VAB creados manualmente")
print(df_vab_manabi)


INGRESAR DATOS VAB MANUALMENTE

```python
datos_vab_manabi = {
    'año': [2007, 2008, ..., 2023],
    'vab_corriente': [valor1, valor2, ..., valorN]
}
df_vab_manabi = pd.DataFrame(datos_vab_manabi)
```

✅ Datos VAB creados manualmente
     año  vab_corriente
0   2007        2077.70
1   2008        2603.60
2   2009        2729.40
3   2010        2969.40
4   2011        3797.40
5   2012        4178.01
6   2013        4369.50
7   2014        4533.80
8   2015        4786.10
9   2016        5061.60
10  2017        4998.03
11  2018        3820.50
12  2019        4898.20
13  2020        4515.50
14  2021        4906.40
15  2022        5496.60
16  2023        5903.50


In [ ]:
# =================================================================
# BLOQUE 10: CALCULAR TASA DE VARIACIÓN DEL VAB
# =================================================================

print("\n" + "=" * 70)
print("CÁLCULO DE TASA DE VARIACIÓN")
print("=" * 70)

# Verifica si df_vab_manabi existe
if 'df_vab_manabi' in locals():

    # Asegurar que esté ordenado por año
    df_vab_manabi = df_vab_manabi.sort_values('año').reset_index(drop=True)

    # Calcular tasa de variación porcentual
    df_vab_manabi['tasa_variacion'] = df_vab_manabi['vab_corriente'].pct_change() * 100

    print("✅ Tasa de variación calculada")
    print("\n📊 VAB DE MANABÍ (2007-2023):")
    print("-" * 70)
    print(df_vab_manabi.to_string(index=False))

    print("\n📈 ESTADÍSTICAS:")
    print(f"  • Tasa de variación promedio: {df_vab_manabi['tasa_variacion'].mean():.2f}%")
    print(f"  • Tasa máxima: {df_vab_manabi['tasa_variacion'].max():.2f}%")
    print(f"  • Tasa mínima: {df_vab_manabi['tasa_variacion'].min():.2f}%")
    print(f"  • Desviación estándar: {df_vab_manabi['tasa_variacion'].std():.2f}%")

else:
    print("⚠️ Primero debes completar el Bloque 8 o 9 para crear df_vab_manabi")


CÁLCULO DE TASA DE VARIACIÓN
✅ Tasa de variación calculada

📊 VAB DE MANABÍ (2007-2023):
----------------------------------------------------------------------
 año  vab_corriente  tasa_variacion
2007        2077.70             NaN
2008        2603.60       25.311643
2009        2729.40        4.831771
2010        2969.40        8.793141
2011        3797.40       27.884421
2012        4178.01       10.022910
2013        4369.50        4.583282
2014        4533.80        3.760156
2015        4786.10        5.564868
2016        5061.60        5.756252
2017        4998.03       -1.255927
2018        3820.50      -23.559883
2019        4898.20       28.208350
2020        4515.50       -7.813074
2021        4906.40        8.656849
2022        5496.60       12.029186
2023        5903.50        7.402758

📈 ESTADÍSTICAS:
  • Tasa de variación promedio: 7.51%
  • Tasa máxima: 28.21%
  • Tasa mínima: -23.56%
  • Desviación estándar: 12.92%


In [ ]:
# =================================================================
# GUARDAR DATASETS DE VAB
# =================================================================

print("\n" + "=" * 70)
print("GUARDADO DE DATASETS VAB")
print("=" * 70)

if 'df_vab_manabi' in locals():

    # Guardar serie completa 2007-2023
    ruta_vab_completo = f"{path_proyecto}/vab_manabi_2007_2023.csv"
    df_vab_manabi.to_csv(ruta_vab_completo, index=False, encoding='utf-8-sig')
    print(f"✅ Serie completa guardada: vab_manabi_2007_2023.csv")
    print(f"   ({len(df_vab_manabi)} años)")

    # Crear subset 2020-2023 para modelo con sentimiento
    df_vab_2020_2023 = df_vab_manabi[df_vab_manabi['año'].between(2020, 2023)].copy()
    ruta_vab_subset = f"{path_proyecto}/vab_manabi_2020_2023.csv"
    df_vab_2020_2023.to_csv(ruta_vab_subset, index=False, encoding='utf-8-sig')
    print(f"✅ Subset 2020-2023 guardado: vab_manabi_2020_2023.csv")
    print(f"   ({len(df_vab_2020_2023)} años)")

    print(f"\n📁 Archivos guardados en Drive:")
    print(f"  • {ruta_vab_completo}")
    print(f"  • {ruta_vab_subset}")

else:
    print("⚠️ No hay datos VAB para guardar")
    print("Completa primero los bloques de extracción de VAB")



GUARDADO DE DATASETS VAB
✅ Serie completa guardada: vab_manabi_2007_2023.csv
   (17 años)
✅ Subset 2020-2023 guardado: vab_manabi_2020_2023.csv
   (4 años)

📁 Archivos guardados en Drive:
  • /content/drive/MyDrive/TitulacionF/vab_manabi_2007_2023.csv
  • /content/drive/MyDrive/TitulacionF/vab_manabi_2020_2023.csv
